---
## 実験: PLS回帰・rawスペクトルデータ使用

In [19]:
import numpy as np
import pandas as pd
import os
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ===== データ読み込み =====
df = pd.read_csv("data/middle/spectrum_bin_mean.csv", encoding='utf-8-sig')

meta_cols = ['sample number', 'species number', '樹種', 'time', '含水率']
feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].values
y = df['含水率'].values

print(f"特徴量数: {len(feature_cols)} | サンプル数: {len(X)}")

特徴量数: 600 | サンプル数: 1322


In [21]:
# ===== 前処理 =====
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

print(f"訓練: {len(X_train)} | 検証: {len(X_val)}")

# ===== n_componentsチューニング =====
n_components_candidates = [30, 40, 50, 60, 70, 80, 90, 100]

print(f"\n== チューニング開始: {len(n_components_candidates)} 候補 ==")
results = []
for n in n_components_candidates:
    pls = PLSRegression(n_components=n)
    pls.fit(X_train_sc, y_train)
    val_rmse = np.sqrt(mean_squared_error(y_val, pls.predict(X_val_sc).flatten()))
    results.append({'n_components': n, 'val_rmse': val_rmse})
    print(f"  n_components={n:3d} -> val RMSE: {val_rmse:.4f}")

results_df = pd.DataFrame(results).sort_values('val_rmse').reset_index(drop=True)
print("\n=== チューニング結果（昇順）===")
print(results_df.to_string())

best_n = int(results_df.iloc[0]['n_components'])
print(f"\n最良パラメータ: n_components={best_n}, val RMSE={results_df.iloc[0]['val_rmse']:.4f}")

訓練: 1057 | 検証: 265

== チューニング開始: 8 候補 ==
  n_components= 30 -> val RMSE: 16.6358
  n_components= 40 -> val RMSE: 13.7276
  n_components= 50 -> val RMSE: 12.6858
  n_components= 60 -> val RMSE: 11.1138
  n_components= 70 -> val RMSE: 10.8836
  n_components= 80 -> val RMSE: 11.0283
  n_components= 90 -> val RMSE: 11.2313
  n_components=100 -> val RMSE: 11.4254

=== チューニング結果（昇順）===
   n_components   val_rmse
0            70  10.883577
1            80  11.028346
2            60  11.113805
3            90  11.231307
4           100  11.425376
5            50  12.685809
6            40  13.727614
7            30  16.635766

最良パラメータ: n_components=70, val RMSE=10.8836


In [22]:
# ===== 最良パラメータで再学習（全訓練データ）=====
print("== ベストパラメータで再学習 ==")
X_all_sc = scaler.fit_transform(X)

best_model = PLSRegression(n_components=best_n)
best_model.fit(X_all_sc, y)

train_rmse = np.sqrt(mean_squared_error(y, best_model.predict(X_all_sc).flatten()))
print(f"最終 train RMSE: {train_rmse:.4f}")

== ベストパラメータで再学習 ==
最終 train RMSE: 5.2507


### 予測値の作成と保存

In [23]:
# ===== テストデータ読み込み =====
df_test = pd.read_csv("data/middle/test_spectrum_bin10.csv", encoding='utf-8-sig')

meta_cols_test = ['sample number', 'species number', 'time', '樹種']
feature_cols_test = [c for c in df_test.columns if c not in meta_cols_test]

X_test = df_test[feature_cols_test].values
X_test_sc = scaler.transform(X_test)
print(f"テストデータの特徴量数: {X_test_sc.shape[1]} | テストサンプル数: {len(X_test_sc)}")

# ===== 予測 =====
preds = best_model.predict(X_test_sc).flatten()

# ===== 提出ファイル保存 =====
save_dir = "data/submission/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "koyama_experiment_pls_v1.csv")

submit = pd.DataFrame({
    'sample_number': df_test['sample number'].values,
    'pred': preds
})
submit.to_csv(save_path, index=False, header=False)
print(f"保存完了: {save_path}")
print(submit.head(10))

テストデータの特徴量数: 600 | テストサンプル数: 550
保存完了: data/submission/koyama_experiment_pls_v1.csv
   sample_number        pred
0             95  199.991727
1             96  160.183828
2             97  146.529726
3             98  151.445241
4             99  156.217019
5            100  143.851526
6            101  146.924931
7            102  159.338074
8            103  151.254876
9            104  149.527578
